# Bronze Layer Ingestion - Provider Data
## Healthcare Lakehouse Project

This notebook ingests the **Provider dataset** (~550 MB) from S3 into the Bronze layer using **Databricks Auto Loader**.

### Key Features:
- **Auto Loader (cloudFiles)**: Efficiently processes new files as they arrive
- **Schema Inference**: Automatically infers schema from CSV files
- **Schema Evolution**: Handles new columns automatically
- **Data Quality Checks**: Row count validation, null checks, duplicate detection
- **Metadata Tracking**: Captures file name and size for lineage
- **Delta Lake**: Writes to Delta format for ACID transactions

### Partitioning Strategy:
- **Bronze**: `read_timestamp` for ingestion batch tracking
- **Silver**: `prscrbr_state_abrvtn` for query optimization (applied in silver layer)

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables
Runs the utilities notebook to access schema name variables (bronze_schema, silver_schema, gold_schema).

In [ ]:
# MAGIC %run /Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/01_setup/utilities

### Step 3: Verify Schema Variables
Confirms that the schema variables are loaded correctly.

In [ ]:
print(bronze_schema, silver_schema, gold_schema)

### Step 4: Define Widget Parameters
Creates interactive widgets for runtime configuration:
- **catalog**: Unity Catalog name (default: healthcare_lakehouse)
- **data_source**: Source system identifier (default: cms)

In [ ]:
dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'cms', 'Data Source')

### Step 5: Get Widget Values
Retrieves the widget values and displays them for verification.

In [ ]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 6: Verify S3 Data Source
Lists the contents of the S3 bucket to confirm data availability.

In [ ]:
display(dbutils.fs.ls("s3://healthcare-claims-lakehouse/raw/cms/"))

### Step 7: Configure Auto Loader Stream
Sets up the Auto Loader configuration:
- **S3_BASE**: Base path to raw CMS data
- **CHECKPOINT_BASE**: Checkpoint location for tracking processed files
- **TARGET_SCHEMA**: Target Bronze schema in Unity Catalog

Auto Loader Configuration:
- **cloudFiles.format**: CSV format
- **maxBytesPerTrigger**: 256MB chunks for optimal processing
- **Schema Location**: Stores inferred schema for consistency
- **Metadata Columns**: Captures file_name and file_size for data lineage

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────
S3_BASE        = f"s3://healthcare-claims-lakehouse/raw/{data_source}"
CHECKPOINT_BASE = f"s3://healthcare-claims-lakehouse/checkpoints/{data_source}"
TARGET_SCHEMA  = f"{catalog}.bronze"

# ── Auto Loader — by_provider (566MB) ────────────────────────────────────
df_provider = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/by_provider/schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.maxBytesPerTrigger", "256m")   # process in 256MB chunks
    .load(f"{S3_BASE}/by_provider/")
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

### Step 8: Data Quality Checks
Validates data before writing to ensure quality and integrity:
- **Row Count**: Validates expected volume
- **Null Checks**: Critical columns (PRSCRBR_NPI, Prscrbr_Last_Org_Name)
- **Duplicate Detection**: Identifies duplicate records by NPI

In [ ]:
# Data Quality Check Function
def validate_provider_data(df):
    """Validate provider ingestion data quality"""
    
    # Row count
    total_rows = df.count()
    print(f"Total rows ingested: {total_rows:,}")
    
    # Null checks on critical columns
    null_checks = df.select([
        F.count(F.when(F.col("PRSCRBR_NPI").isNull(), 1)).alias("null_prscrbr_npi"),
        F.count(F.when(F.col("Prscrbr_Last_Org_Name").isNull(), 1)).alias("null_last_org_name"),
        F.count(F.when(F.col("Prscrbr_City").isNull(), 1)).alias("null_city"),
        F.count(F.when(F.col("Prscrbr_State_Abrvtn").isNull(), 1)).alias("null_state")
    ])
    print("Null checks on critical columns:")
    null_checks.show(truncate=False)
    
    # Duplicate detection by NPI
    duplicates = df.groupBy("PRSCRBR_NPI").count().filter("count > 1")
    dup_count = duplicates.count()
    print(f"Duplicate NPI records found: {dup_count:,}")
    
    # File metadata
    print("\nFile metadata:")
    df.select("file_name", "file_size", "read_timestamp").distinct().show(truncate=False)
    
    # State distribution
    print("\nTop 10 states by record count:")
    df.groupBy("Prscrbr_State_Abrvtn").count().orderBy(F.desc("count")).limit(10).show()
    
    return total_rows, dup_count

# Run validation (batch mode for preview)
# Note: For streaming, validation happens per micro-batch

### Step 9: Write to Bronze Delta Table
Writes the streaming data to the Bronze layer Delta table:
- **Partition By**: `read_timestamp` for ingestion batch tracking
  - Silver layer repartitions by `prscrbr_state_abrvtn` for query optimization
- **Checkpoint Location**: Tracks processing progress
- **mergeSchema**: Allows schema evolution
- **outputMode**: Append mode for incremental loads
- **trigger.availableNow**: Processes all available data and stops (batch-like behavior)

In [ ]:
# ── Write to Bronze Delta table ───────────────────────────────────────────
(
    df_provider.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/by_provider/checkpoint")
    .option("mergeSchema", "true")
    .partitionBy("read_timestamp")  # Bronze: ingestion batch tracking
    .outputMode("append")
    .trigger(availableNow=True)  # runs once, processes all data, then stops
    .toTable(f"{TARGET_SCHEMA}.by_provider")
)

### Step 10: Preview Data and Validation Results (Batch Mode)
Reads the data in batch mode to preview the ingested records and run validation.

In [ ]:
# Preview data by reading in batch mode
df_preview = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_BASE}/by_provider/")
)

In [ ]:
display(df_preview.limit(20))

In [ ]:
# Run data quality validation
total_rows, dup_count = validate_provider_data(df_preview)
print(f"\n✅ Validation Complete: {total_rows:,} rows, {dup_count:,} duplicate NPIs")

### Step 11: Cleanup Checkpoint (Optional)
Removes checkpoint directory if re-running from scratch.

In [ ]:
dbutils.fs.rm(f"{CHECKPOINT_BASE}/by_provider", True)

### Step 12: Alternative Write with awaitTermination
Alternative approach using awaitTermination() for blocking execution.

In [ ]:
query = (
    df_provider.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/by_provider/checkpoint")
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{TARGET_SCHEMA}.by_provider")
)

query.awaitTermination()